# **Understanding Data**

In [ ]:
import pandas as pd


In [ ]:
df=pd.read_csv('merged_data.csv',low_memory=False,encoding='utf-8')
print(df.shape)

In [ ]:
print(df.dtypes.value_counts())
print(df.duplicated().sum())
print(df['Label'].value_counts())

In [ ]:
df.describe().T

# **Exploratory Data Analysis (EDA)**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['Flow Duration'], bins=50, ax=axes[0])
axes[0].set_title('Distribution — Flow Duration')
sns.boxplot(x=df['Flow Duration'], ax=axes[1])
axes[1].set_title('Boxplot — Flow Duration')
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols].skew().sort_values(ascending=False).head(20)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Label', y='Init_Win_bytes_forward')
plt.xticks(rotation=45, ha='right')
plt.title('Init_Win_bytes_forward by Attack Type')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_temp = LabelEncoder()
y_temp = le_temp.fit_transform(df['Label'])

corr_with_target = df[numeric_cols].corrwith(pd.Series(y_temp, index=df.index))
corr_with_target.abs().sort_values(ascending=False).head(20)

In [ ]:
import numpy as np

corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),
k=1).astype(bool))

high_corr_pairs = upper.stack()
high_corr_pairs = high_corr_pairs[high_corr_pairs > 0.97]
print(high_corr_pairs.sort_values(ascending=False))

# **Structural Cleaning**

In [ ]:
zero_variance_cols = [
    col for col in df.select_dtypes(include='number').columns
    if df[col].nunique() <= 1
]
print(zero_variance_cols)

df = df.drop(columns=zero_variance_cols)

In [ ]:
duplicate_cols_to_drop=[
    'Subflow Fwd Packets',        # duplicates Total Fwd Packets
    'Subflow Bwd Packets',        # duplicates Total Backward Packets
    'Subflow Fwd Bytes',          # duplicates Total Length of Fwd Packets
    'Subflow Bwd Bytes',          # duplicates Total Length of Bwd Packets
    'Avg Fwd Segment Size',       # duplicates Fwd Packet Length Mean
    'Avg Bwd Segment Size',       # duplicates Bwd Packet Length Mean
    'Fwd Header Length.1',        # exact duplicate of Fwd Header Length
                                  # (also duplicates Bwd Header Length —
                                  #  all three are identical in this data)
    'CWE Flag Count',             # duplicates Fwd URG Flags
    'ECE Flag Count',             # duplicates RST Flag Count
    'SYN Flag Count',             # duplicates Fwd PSH Flags —
    ]

duplicate_cols_to_drop=[c for c in duplicate_cols_to_drop
                        if c in df.columns]
df=df.drop(columns=duplicate_cols_to_drop)
print(f'Dropped {len(duplicate_cols_to_drop)} duplicate cols, {df.shape[1]} remain')


In [ ]:
df

In [ ]:
df['Label']=df['Label'].astype(str).str.strip()
df['Label']=df['Label'].str.replace('\ufffd','-',regex=False)

print(sorted(df['Label'].unique()))

In [ ]:
df[df.duplicated()]['Label'].value_counts(normalize=True)

In [ ]:
before=len(df)
df=df.drop_duplicates()
after=len(df)
print(f'Removed {before - after} duplicate rows ')
print(f'Remaining: {after:,} rows')

In [ ]:
print(f'Started with: 79 columns')
print(f'After dropping zero-variance: {79 - 8} columns')
print(f'After dropping duplicates: {79 - 8 - 10} columns')
print(f'Actual remaining: {df.shape[1]} columns')
print(f'Actual remaining: {df.shape[0]} rows')

# **Handling Missing Values**

In [ ]:
inf_counts=np.isinf(df.select_dtypes(include='number')).sum()
print(inf_counts[inf_counts > 0])

df=df.replace([np.inf,-np.inf],np.nan)

In [ ]:
missing=df.isnull().sum()
print(missing[missing>0])


rows_with_any_na=df.isnull().any(axis=1).sum()
print(f'{rows_with_any_na:,} rows affected ({rows_with_any_na/len(df)*100:.4f} % of data)')

In [ ]:
missing_mask=df.isnull().any(axis=1)
print(df.loc[missing_mask,'Label'].value_counts())
print()
print(df.loc[missing_mask,'Label'].value_counts(normalize=True)*100)



*   **Missingness Concentrates**
*   **Decision: impute, don't drop**







In [ ]:
from sklearn.impute import SimpleImputer

rate_cols=['Flow Bytes/s', 'Flow Packets/s']
imputer=SimpleImputer(strategy='median')
df[rate_cols]=imputer.fit_transform(df[rate_cols])

In [ ]:
before = len(df)
df = df[df['Flow Duration'] >= 0]
after = len(df)
print(f'Removed {before - after:,} rows with impossible negative duration')

# **Outlier Detection and Treatment**

In [ ]:
def iqr_bounds(series,k=1.5):
  Q1,Q3=series.quantile(0.25),series.quantile(0.75)
  IQR=Q3-Q1
  return Q1-k*IQR,Q3+k*IQR

for col in ['Flow Duration','Flow Bytes/s','Flow Packets/s','Fwd Packet Length Max']:
  lower,upper=iqr_bounds(df[col])
  mask=(df[col]<lower)|(df[col]>upper)
  print(f'{col}: bounds=[{lower:.1f}, {upper:.1f}, outliers={mask.sum():,} ({mask.mean()*100:.2f}%)]')
  outliers = df[(df['Flow Duration'] < lower) | (df['Flow Duration'] > upper)]
  print(outliers['Label'].value_counts(normalize=True).head(5) * 100)


In [ ]:
def cap_outliers_in_benign_only(df, col, k=1.5):
    benign_mask = df['Label'] == 'BENIGN'
    lower, upper = iqr_bounds(df.loc[benign_mask, col], k=k)
    df.loc[benign_mask, col] = df.loc[benign_mask, col].clip(lower=lower,
upper=upper)
    return df
for col in ['Flow Duration', 'Flow Bytes/s', 'Flow Packets/s', 'Fwd Packet Length Max']:
    df = cap_outliers_in_benign_only(df, col)

In [ ]:
from sklearn.model_selection import train_test_split

x=df.drop(columns=['Label'])
y=df['Label']

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
print()

 # **Encoding Categorical Data**

In [ ]:
label_map = {
    'BENIGN':                  'Normal Traffic',
    'DoS GoldenEye':           'DoS',
    'DoS Hulk':                'DoS',
    'DoS Slowhttptest':        'DoS',
    'DoS slowloris':           'DoS',
    'DDoS':                    'DDoS',
    'PortScan':                'Port Scanning',
    'FTP-Patator':             'Brute Force',
    'SSH-Patator':             'Brute Force',
    'Bot':                     'Bots',
    'Web Attack - Brute Force':'Web Attacks',
    'Web Attack - Sql Injection':'Web Attacks',
    'Web Attack - XSS':        'Web Attacks',
    'Infiltration':            'Infiltration',
    'Heartbleed':              'Heartbleed',
}
df['Label'] = df['Label'].map(label_map).fillna(df['Label'])
# .replace() automatically leaves any values not in the dictionary exactly as they were
#df['Label'] = df['Label'].replace(label_map)

In [ ]:
print(df['Label'].value_counts())

In [ ]:
# enumerate() pairs the array index (0, 1, 2...) with the label string
#label_classes = dict(enumerate(le.classes_))

#print(label_classes)
# Output: {0: 'Bots', 1: 'Brute Force', 2: 'DDoS', ...}

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Label_Encoded'] = le.fit_transform(df['Label'])
# Keep this mapping — you need it to translate predictions back to
# human-readable names in the dashboard
label_classes = dict(zip(le.transform(le.classes_), le.classes_))
print(label_classes)
# e.g. {0: 'Bots', 1: 'Brute Force', 2: 'DDoS', 3: 'DoS', ...}

 # **Feature Construction & Splitting**

In [ ]:
# Feature 1 — Packet Direction Ratio

df['Pkt_Ratio'] = df['Total Backward Packets'] / (df['Total Fwd Packets'] + 1)
print(df.groupby('Label')['Pkt_Ratio'].median().sort_values())
# Measured result — the hypothesis only partly held up

In [ ]:
#  Bytes Per Packet (payload density)

df['Bytes_Per_Pkt'] = df['Flow Bytes/s'] / (df['Flow Packets/s'] + 1)

In [ ]:
#  Inter-Arrival-Time Regularity

df['IAT_Regularity'] = df['Flow IAT Std'] / (df['Flow IAT Mean'].abs() + 1)

In [ ]:
# Flag Density

flag_cols = ['FIN Flag Count', 'Fwd PSH Flags', 'PSH Flag Count',
             'ACK Flag Count', 'URG Flag Count']
total_pkts = df['Total Fwd Packets'] + df['Total Backward Packets'] + 1
df['Flag_Density'] = df[flag_cols].sum(axis=1) / total_pkts

# **Column Transformer & Pipelines**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
rate_cols = ['Flow Bytes/s', 'Flow Packets/s']
all_numeric_cols = x.columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ('impute_rates', SimpleImputer(strategy='median'), rate_cols),
    ('scale_all', RobustScaler(), all_numeric_cols),
], remainder='passthrough')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf_pipeline=Pipeline(steps=[('preprocessing',preprocessor),
                            ('classifier',RandomForestClassifier
                             (n_estimators=100,
                            class_weight='balanced',max_depth=20,
                            min_samples_leaf=2,n_jobs=-1,random_state=42))])

rf_pipeline.fit(x_train,y_train)
predictions=rf_pipeline.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(predictions,y_test)


In [ ]:

from imblearn.ensemble import BalancedRandomForestClassifier
from collections import Counter
y_counts = Counter(y_train)

print("Training class sizes:")
for label, count in sorted(y_counts.items(), key=lambda x: x[1]):
    print(f"  {label:<20} {count:>10,}")

# Use 500 as the target — large enough for trees to learn real patterns,
# small enough that Infiltration (28 rows) and Heartbleed (8 rows)
# are never asked for more than they have.
TARGET_PER_CLASS = 500

sampling_strategy = {
    cls: min(count, TARGET_PER_CLASS)
    for cls, count in y_counts.items()
}

print("\nSampling strategy (rows per class per tree):")
for label, size in sorted(sampling_strategy.items(), key=lambda x: x[1]):
    print(f"  {label:<20} {size:>6,}")


rf_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', BalancedRandomForestClassifier(
        n_estimators=200,          # 200 trees — more diversity since each
                                   # tree sees limited data per class
        sampling_strategy=sampling_strategy,
        replacement=True,          # with replacement — preserves tree
                                   # diversity for tiny classes like
                                   # Heartbleed (8 rows in training)
        max_depth=20,              # prevents extreme overfitting
        min_samples_leaf=1,        # can be 1 — no synthetic data,
                                   # no risk of memorising a fake point
        max_features='sqrt',       # each split considers sqrt(n_features)
                                   # columns — standard for RF, adds
                                   # more diversity between trees
        class_weight=None,         # NOT 'balanced' — BRF already handles
                                   # imbalance via sampling. Double-
                                   # correcting makes rare classes
                                   # over-dominant
        n_jobs=-1,
        random_state=42,
    )),
])


print("\nTraining Balanced Random Forest...")
rf_pipeline.fit(x_train, y_train)
print("✅ Done")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = rf_pipeline.predict(x_test)

# ── Classification report ─────────────────────────────────────────
# This is the only metric that matters for your imbalanced dataset.
# Read the per-class F1 for Web Attacks, Bots, Infiltration, Heartbleed.
# These four are where the model either works or doesn't.
print(classification_report(y_test, y_pred, zero_division=0))

# ── Macro F1 — the honest single number ───────────────────────────
from sklearn.metrics import f1_score
macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
print(f"Macro F1 (weights all classes equally): {macro_f1:.4f}")
# This penalises poor performance on Heartbleed just as much as on
# Normal Traffic — which is exactly what you want for a NIDS.

# ── Confusion matrix ──────────────────────────────────────────────
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title('Confusion Matrix — Balanced Random Forest')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import joblib, os
os.makedirs('models', exist_ok=True)

# The RF pipeline (preprocessor + classifier in one object)
joblib.dump(rf_pipeline, 'models/rf_pipeline.pkl')

# The sampling strategy — useful to log for reproducibility
joblib.dump(sampling_strategy, 'models/sampling_strategy.pkl')

print("Saved:")
print("  models/rf_pipeline.pkl")